<a href="https://colab.research.google.com/github/uddeeshmeela/LLM-Projects/blob/main/Building_AI_Agents_with_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('api_key')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [ ]:
!pip install langchain langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch
from google.colab import userdata

tavily_api_key = userdata.get('TAVILY_API_KEY')
skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)

result = skill_demand_tool.invoke({"query": "generative ai skills demand 2025"})
print(result)


{'query': 'generative ai skills demand 2025', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://aws.amazon.com/executive-insights/content/top-generative-ai-skills-and-education-trends-for-2025/', 'title': 'Top Generative AI Skills and Education Trends for 2025 - AWS', 'content': 'AI skills in just one year – as part of Amazon’s AI Ready commitment. The demand for AI skills training will only continue to rise. [...] skills to complete the project or initiative so you don’t have to recruit new talent, outsource the work, or worse, shelf the project altogether. Generative AI skills will absolutely make employees more productive, giving you the ability to add new responsibilities to their scope when other, more automated tasks fall away. Measuring the business impact of ongoing training comes down to what can be accomplished that was not possible without the skills development. This takes into account your teams’ engagement, retention, efficiency, coll

In [ ]:
import requests
from google.colab import userdata

rapidapi_key = userdata.get('RAPIDAPI_KEY')

url = "https://jsearch.p.rapidapi.com/search"
headers = {
  "x-rapidapi-key": rapidapi_key,
  "x-rapidapi-host": "jsearch.p.rapidapi.com"
}
query_string = {
  "query": "Generative AI in India",
  "page": "1",
  "num_pages": "1"
}

response = requests.get(url, headers=headers, params=query_string)
data = response.json()
print(data)


{'status': 'OK', 'request_id': 'ca89572b-7398-4515-b4d1-e5b662b1ba32', 'parameters': {'query': 'Generative AI in India', 'page': 1, 'num_pages': 1, 'country': 'us', 'language': 'en'}, 'data': [{'job_id': 'E_Z53nXHcGGCcJ_iAAAAAA==', 'job_title': 'Senior Research Scientist, Generative AI', 'employer_name': 'Uber', 'employer_logo': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ5P-JaTkhLgASUEADjcI9_THrw2ftb9JylguhC&s=0', 'employer_website': 'https://www.uber.com', 'job_publisher': 'Uber', 'job_employment_type': 'Full-time', 'job_employment_types': ['FULLTIME'], 'job_apply_link': 'https://www.uber.com/careers/list/157170', 'job_apply_is_direct': False, 'apply_options': [{'apply_link': 'https://www.uber.com/careers/list/157170', 'is_direct': False, 'publisher': 'Uber'}], 'job_description': "About the Role\n\nUber AI Solutions is one of Uber’s biggest bets with the ambition to build one of the world’s largest data foundries for AI applications and evolve into a platform of choice fo

In [ ]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
  """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
  print(f"\nCalling search_jobs tool")
  print(f"Searching jobs for: {skill} in {location}")

  rapidapi_key = userdata.get('RAPIDAPI_KEY')

  url = "https://jsearch.p.rapidapi.com/search"
  headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
  }
  querystring = {
    "query": f"{skill} in {location}",
    "page": "1",
    "country": "in",
    "employment_types": "INTERN,FULLTIME",
    "job_requirements": "no_experience,under_3_years_experience"
  }
  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()
  jobs = data.get("data", [])
  print(f"Found {len(jobs)} jobs\n")

  result = []
  for job in jobs:
    result.append({
      "title": job.get("job_title"),
      "company": job.get("employer_name"),
      "location": job.get("job_city"),
      "apply_link": job.get("job_apply_link")
    })
  return result


In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
  model=model,
 tools=[skill_demand_tool, search_jobs],
 system_prompt=system_prompt,
  checkpointer=checkpointer,
 #debug=True

)
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
  "messages": [{"role": "user", "content": user_query}]
},config=config)
print(response["messages"][-1].content)




Calling search_jobs tool
Searching jobs for: generative ai in India
Found 10 jobs

[{'type': 'text', 'text': 'Here\'s an overview of the demand for Generative AI in the industry and related job openings in India:\n\nIndustry Demand for Generative AI:\n\n  The demand for generative AI skills has seen substantial growth, with unique job postings increasing from 55 in January 2021 to nearly 10,000 by May 2025. This acceleration notably began in early 2023, coinciding with the widespread adoption of tools like ChatGPT. The emergence of "Generative Artificial Intelligence Engineer" as a distinct occupation further highlights this trend.\n\n  Generative AI skills are no longer confined to traditional tech roles. While Data Scientists and Machine Learning Engineers continue to lead demand, the need for these skills has expanded to roles such as Solutions Architects, Product Managers, and Enterprise Architects, indicating that AI capabilities are being integrated across various business funct

In [ ]:
print(response["messages"][-1].content[0]['text'])

Generative AI Industry Demand and Job Opportunities in India

Industry Demand for Generative AI:

  Generative AI is experiencing significant and sustained growth across various industries.
  The technology industry leads in adoption, with 56% of projects, often developing generative AI products sold to other companies.
  Other major industries adopting generative AI include manufacturing (8%), professional services (7%), financial services, healthcare, retail, e-commerce, media, and entertainment.
  Generative AI is being used to augment various business activities, with customer issue resolution being the most common (35% of projects).
  Gartner projects that over 100 million people will use generative AI for work by 2026, and McKinsey estimates it could add $2.6 trillion to $4.4 trillion to the global economy.
  Spending on generative AI is rapidly increasing, with companies spending an estimated $37 billion in 2025, up from $11.5 billion in 2024.
  Demand for generative AI skills h